# NammaRetina: reproducible GPU training
Select **Runtime → Change runtime type → T4 GPU** in Google Colab. This notebook downloads the versioned 3,662-image APTOS derivative, audits exact duplicates, trains with the corrected EfficientNet input range, selects on validation accuracy, and exports a production-compatible candidate. It cannot promise a particular score. The fixed test partition is evaluated only at the end. Dataset terms and source provenance remain applicable; the mirror is resized, not a second independent dataset.

Use a runtime version with Python 3.11 or 3.12. If the default is newer, choose an available past runtime in Runtime → Change runtime type; [official instructions](https://research.google.com/colaboratory/runtime-version-faq.html).


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/content/NammaRetina')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/The-ChandanKV/NammaRetina.git', str(repo)], check=True)
%cd /content/NammaRetina


In [ ]:
import sys
assert sys.version_info[:2] in {(3, 11), (3, 12)}, 'Select a past Colab runtime with Python 3.11/3.12 under Runtime > Change runtime type; see https://research.google.com/colaboratory/runtime-version-faq.html'
# Install before importing TensorFlow; restart the runtime if requested.
%pip install -r training/requirements.txt


In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Enable a GPU runtime first'


In [ ]:
!python -m training.download_data


## Train and evaluate
The frozen-head warmup runs first, then the top convolutional layers are fine-tuned with batch normalization frozen. A single best-validation threshold protects the stronger warmup checkpoint if fine-tuning gets worse. For development experiments add `--no-test`; inspect the held-out test results only after choosing the final recipe. No accuracy target is fabricated and no model is deployed automatically.


In [ ]:
!python -m training.train --data-root training/data/aptos224 --run-dir training/outputs/gpu_candidate --mode finetune --require-gpu --head-epochs 40 --fine-epochs 20 --batch-size 32


In [ ]:
import json
from pathlib import Path
run = Path('training/outputs/gpu_candidate')
results = json.loads((run / 'metrics.json').read_text())
print(json.dumps(results, indent=2))
print('SHA256:', (run / 'model.sha256').read_text())
assert results['production_smoke_test']['gradcam']


In [ ]:
# Persist artifacts in your own Drive; do not upload the patient dataset to Git.
from google.colab import drive
import shutil
drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/NammaRetina/gpu_candidate')
destination.mkdir(parents=True, exist_ok=True)
for name in ['efficientnetb0_dr.keras', 'metrics.json', 'model_metadata.json', 'model.sha256', 'data_audit.json', 'test_predictions.csv']:
    shutil.copy2(run / name, destination / name)
print('Saved to', destination)


## Put the selected candidate into production
Review all five recalls, macro-F1, QWK, undergrading counts and the accuracy interval in `metrics.json`. The previous 75.41% notebook result is not directly comparable because no old artifact exists and duplicate quarantine changes membership. Clinical generalization is not established by this experiment.

Host `efficientnetb0_dr.keras` at a stable HTTPS download URL (or a Google Drive file link with download access). In Render set `MODEL_URL` and set `MODEL_SHA256` to the digest in `model.sha256`, then deploy/restart. Do not commit a `.keras` file to Git. Verify `/health` reports `model_available: true` and the exact `model_sha256`. Database readiness is separate: verify `database_available: true` too, and repair Supabase configuration if it is false. Keep the previous URL/checksum for rollback.
